In [1]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
import glob

from limb_fitting import *
from ellipse import *
from utils import *

In [2]:
def stretch(x, y, amount=1, angle=0, center=(0,0)):
    xc, yc = center
    r = amount
    a = angle
    R = np.array([[1 + np.cos(2 * a), np.sin(2 * a), -((1 + np.cos(2 * a)) * xc + np.sin(2 * a) * yc)],
                  [np.sin(2 * a), 1 - np.cos(2 * a), -(np.sin(2 * a) * xc + (1 - np.cos(2 * a)) * yc)],
                  [0, 0, 0]])
    R = np.identity(3) + (r - 1) / 2 * R
    xn = R[0,0] * x + R[0,1] * y + R[0,2]
    yn = R[1,0] * x + R[1,1] * y + R[1,2]
    return xn, yn

def undistort(x, y, coefficient=0, center=(0,0)):
    xc, yc = center
    k = coefficient

    r2 = (x - xc) ** 2 + (y - yc) ** 2
    q = 1 / (1 + k * r2)
    return xc + (x - xc) * q, yc + (y - yc) * q

def distort(x, y, coefficient=0, center=(0,0)):
    xc, yc = center
    k = coefficient

    r2 = (x - xc) ** 2 + (y - yc) ** 2
    q = (1 - np.sqrt(1 - 4 * k * r2)) / (2 * k * r2 + 1e-16)
    return xc + (x - xc) * q, yc + (y - yc) * q

In [3]:
s = np.load('/home/ulyanov/data/solo/phi/distortion/fdt/residual_20240319.npz')
dx, dy = s['dx'], s['dy']

In [7]:
files = sorted(glob.glob('/home/ulyanov/data/solo/phi/flat/fdt/calibration/2025-09-23/*.fits.gz'))
files

['/home/ulyanov/data/solo/phi/flat/fdt/calibration/2025-09-23/solo_L1_phi-fdt-ilam_20250923T000503_V202512011636C_0569230100.fits.gz',
 '/home/ulyanov/data/solo/phi/flat/fdt/calibration/2025-09-23/solo_L1_phi-fdt-ilam_20250923T001105_V202511212307C_0569230125.fits.gz',
 '/home/ulyanov/data/solo/phi/flat/fdt/calibration/2025-09-23/solo_L1_phi-fdt-ilam_20250923T001705_V202511212307C_0569230150.fits.gz',
 '/home/ulyanov/data/solo/phi/flat/fdt/calibration/2025-09-23/solo_L1_phi-fdt-ilam_20250923T002305_V202511212307C_0569230175.fits.gz',
 '/home/ulyanov/data/solo/phi/flat/fdt/calibration/2025-09-23/solo_L1_phi-fdt-ilam_20250923T002905_V202511212307C_0569230200.fits.gz',
 '/home/ulyanov/data/solo/phi/flat/fdt/calibration/2025-09-23/solo_L1_phi-fdt-ilam_20250923T003505_V202511212307C_0569230225.fits.gz',
 '/home/ulyanov/data/solo/phi/flat/fdt/calibration/2025-09-23/solo_L1_phi-fdt-ilam_20250923T004105_V202511212338C_0569230250.fits.gz',
 '/home/ulyanov/data/solo/phi/flat/fdt/calibration/2025

In [5]:
xi, yi = np.mgrid[:2048,:2048].astype(np.float32)

xi -= dx
yi -= dy

xd, yd = distort(xi, yi, -2e-8, (937, 1055))
xd, yd = stretch(xd, yd, 1.001, -0.567, (1023.5, 1023.5)) ### Distorted grid

dxd = xd - xi, yd - yi

In [8]:
for file in files[:]:

    with fits.open(file) as hdul:
        image  = hdul[0].data[0]
        header = hdul[0].header

    image = bilinear(image, xd, yd)

    edges = find_edges(image)
    x, y = np.where(edges)
    x, y = filter_outliers(x, y, acc=1)
    ellipse = fit_ellipse(x, y)

    print(ellipse.radius, ellipse.flatness)

814.1855859052646 0.00010537448881919875
814.2247128014991 0.00043315266303378674
814.050493786166 0.00018868367567947963
813.8506302674562 0.00016251309095960398
813.3956744712414 0.0008401909192690127
813.5582242211044 0.0002794139486383962
813.5426059483864 2.7615428181970714e-05
813.5414320599941 0.00020438019935931084
813.485328196377 0.000637962986779983


In [9]:
xi, yi = np.mgrid[:2048,:2048].astype(np.float32)

xu, yu = stretch(xi, yi, 1 / 1.001, -0.567, (1023.5, 1023.5))
xu, yu = undistort(xu, yu, -2e-8, (937, 1055)) ### Undistorted grid

xu += dx
yu += dy

dxu, dyu = xu - xi, yu - yi

In [10]:
for file in files[:]:

    with fits.open(file) as hdul:
        image  = hdul[0].data[0]
        header = hdul[0].header

    edges = find_edges(image)
    x, y = xu[edges], yu[edges]
    x, y = filter_outliers(x, y, acc=1)
    ellipse = fit_ellipse(x, y)

    print(ellipse.radius, ellipse.flatness)

814.191538559126 7.974382619679865e-05
814.2247910904973 0.00044560686206951416
814.0633853157104 0.00018020348224911942
813.8657055845202 0.0001454290309856887
813.4296095605146 0.0008062885997907054
813.57478566035 0.00025936756287725515
813.5657868341641 4.077678371017246e-05
813.5476134517465 0.00019668137949591102
813.5040581086475 0.0006075368732715303


In [11]:
np.savez('distortion_cor.npz', xd=xd, yd=yd, xu=xu, yu=yu)

In [18]:
files = sorted(glob.glob('/home/ulyanov/data/cross/*stokes*'))
file = files[0]

In [21]:
with fits.open(file) as hdul:
    image  = hdul[0].data[0,0]
    header = hdul[0].header

nx, ny = header['NAXIS2'], header['NAXIS1']
x0, y0 = header['PXBEG2'] - 1, header['PXBEG1'] - 1
xd_, yd_ = xd[x0:x0+nx, y0:y0+ny] - x0, yd[x0:x0+nx, y0:y0+ny] - y0
xu_, yu_ = xu[x0:x0+nx, y0:y0+ny] - x0, yu[x0:x0+nx, y0:y0+ny] - y0

edges = find_edges(image)
#x, y = np.where(edges)
x, y = xu_[edges], yu_[edges]
x, y = filter_outliers(x, y, acc=1)
ellipse = fit_ellipse(x, y)

ellipse.center, ellipse.radius, ellipse.flatness

((np.float64(678.7746670652833), np.float64(720.4209558597424)),
 np.float64(613.0908465231014),
 np.float64(0.0001935840739262229))